In [2]:
class Term:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name

# Predicates that link Terms to FOL expressions (like relationships)
class Predicate:
    def __init__(self, name):
        self.name = name
        self.facts = []

    def add_fact(self, *args):
        self.facts.append(args)

    def check(self, *args):
        return args in self.facts

    def __repr__(self):
        return f"{self.name}: {self.facts}"

# Example of basic inference rule (diagnosing disease based on symptoms)
class Rule:
    def __init__(self, if_clause, then_clause):
        self.if_clause = if_clause
        self.then_clause = then_clause

    def apply(self):
        inferred_facts = []
        for fact in self.if_clause.facts:
            if self.if_clause.check(*fact):
                inferred_facts.append(self.then_clause(*fact))
        return inferred_facts


# Define the Terms (Patients, Symptoms, Diseases)
cough = Term("Cough")
fever = Term("Fever")
loss_of_taste = Term("Loss of Taste")
body_aches = Term("Body Aches")
fatigue = Term("Fatigue")

# Diseases
flu = Term("Flu")
covid19 = Term("Covid19")

# Treatments
rest_and_hydration = Term("Rest and Hydration")
isolation_and_medication = Term("Isolation and Medication")

# Define the Predicates (relationships between terms)
#has_symptom = Predicate("HasSymptom")
causes = Predicate("Causes")
treatment_for = Predicate("TreatmentFor")

# Add facts: what diseases cause what symptoms
causes.add_fact(flu, cough)
causes.add_fact(flu, fever)
causes.add_fact(flu, body_aches)
causes.add_fact(flu, fatigue)

causes.add_fact(covid19, cough)
causes.add_fact(covid19, fever)
causes.add_fact(covid19, loss_of_taste)
causes.add_fact(covid19, fatigue)

# Add facts: what treatments are associated with which diseases
treatment_for.add_fact(flu, rest_and_hydration)
treatment_for.add_fact(covid19, isolation_and_medication)

# Modified rule to diagnose based on partial symptoms match
def diagnose_rule(patient_symptoms, disease):
    disease_symptoms = [fact[1] for fact in causes.facts if fact[0] == disease]
    matching_symptoms = [symptom for symptom in patient_symptoms if symptom in disease_symptoms]

    # If at least one symptom matches, consider it a possible diagnosis
    if matching_symptoms:
        return disease, matching_symptoms
    return None, []

# Rule for suggesting treatment (based on diagnosed disease)
def treatment_rule(disease):
    treatment = [fact[1] for fact in treatment_for.facts if fact[0] == disease]
    if treatment:
        return treatment[0]
    else:
        return "No treatment available"

# Inference Engine
def inference_engine(patient_symptoms):
    #Diagnose the patient based on partial symptom matches
    diagnosed_diseases = []
    for disease in [flu, covid19]:
        diagnosis, matching_symptoms = diagnose_rule(patient_symptoms, disease)
        if diagnosis:
            diagnosed_diseases.append((diagnosis, matching_symptoms))

    #Suggest treatment based on the diagnosed diseases
    treatments = []
    for disease, _ in diagnosed_diseases:
        treatment = treatment_rule(disease)
        treatments.append((disease, treatment))

    return diagnosed_diseases, treatments

# User input to add symptoms
def get_user_symptoms():
    print("Enter the symptoms the patient has (comma separated):")
    input_symptoms = input().split(',')
    input_symptoms = [symptom.strip().lower() for symptom in input_symptoms]  # Convert to lowercase for uniformity

    valid_symptoms = {
        "cough": cough,
        "fever": fever,
        "loss of taste": loss_of_taste,
        "body aches": body_aches,
        "fatigue": fatigue
    }

    patient_symptoms = []

    for symptom in input_symptoms:
        if symptom in valid_symptoms:
            patient_symptoms.append(valid_symptoms[symptom])
        else:
            print(f"Unrecognized symptom: {symptom}")

    if not patient_symptoms:
        print("No valid symptoms entered.")
        return []

    return patient_symptoms

# Main function to run the diagnosis system
def main():
    # Get user symptoms
    patient_symptoms = get_user_symptoms()

    if not patient_symptoms:
        return

    # Run the inference engine
    diagnosed_diseases, suggested_treatments = inference_engine(patient_symptoms)

    # Output the diagnosis
    print("\nDiagnosis Results:")
    if diagnosed_diseases:
        for disease, matching_symptoms in diagnosed_diseases:
            print(f"- Possible diagnosis: {disease.name} (matching symptoms: {', '.join([s.name for s in matching_symptoms])})")
    else:
        print("No diagnosis could be made.")

    # Output the suggested treatments
    print("\nSuggested Treatments:")
    if suggested_treatments:
        for disease, treatment in suggested_treatments:
            print(f"- For {disease.name}: {treatment.name}")
    else:
        print("No treatments found.")

# Run the program
if __name__ == "__main__":
    main()

Enter the symptoms the patient has (comma separated):

Diagnosis Results:
- Possible diagnosis: Flu (matching symptoms: Cough)
- Possible diagnosis: Covid19 (matching symptoms: Cough)

Suggested Treatments:
- For Flu: Rest and Hydration
- For Covid19: Isolation and Medication
